In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import re

from sklearn.pipeline import Pipeline
from sklearn.cluster import DBSCAN
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier

In [2]:
log_files = glob.glob('/tmp/logs/*.log')

In [34]:
    
common_messages = [
    r'Accepted',
    r'pam_unix\(\w+\:\w+\)',
    r'Failed',
    r'Received',
    r'Invalid user',
    r'Authentication failure',
    r'refused connect from',
    r'error: maximum authentication attempts exceeded',
    r'Connection closed by',
    r'New session',
    r'Disconnected'
]


cm_pattern = r'\s({})'.format('|'.join(common_messages))

pattern_step1 = r'(\w+\s\d+\s\d{2}:\d{2}:\d{2})\s(\w+)\s(\w+)(\[(\d+)\]|):' + cm_pattern

messages = [
    r'Invalid user\s(\w+)\s\w+\s(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\s\w+\s(\d+)',
    r'pam_unix\(sshd:session\): session\s(\w+)\s\w+\s\w+\s(\w+)(\((\w+=\d+)\)\s\w+\s\((\w+=\d+)\)|)',
       
]

pattern_step2 = r'\s({})'.format('|'.join(messages))
log_dict = {}
log_list = []

for file_path in log_files:
    with open(file_path, 'r') as file:    
    # Lire chaque ligne du fichier auth.log
        for line in file.readlines():
            print(line)
            
            # Rechercher les correspondances dans chaque ligne avec le modèle d'expression régulière
            match_step1 = re.search(pattern_step1, line)
            if match_step1:
                # Extraire les informations spécifiques à l'aide des groupes capturés
                log_dict = {
                    'Timestamp': match_step1.group(1),
                    'Node' : match_step1.group(2),
                    'Event' : match_step1.group(3),
                    'sshd_id' : match_step1.group(5),
                    'common_messages' : match_step1.group(6),
                    'type_session': '',
                    'type_accepted': '',
                    'type_received': '',
                    'type_connection': '',
                    'type_fail': '',
                    'type_auth': '',
                    'user':'',
                    'IP_user' : '',
                    'port_user': '',
                    'logname':'',
                    'uid_target': '',
                    'uid_init' : '',
                    'euid' : '',
                    'tty' : '',
                    'ruser' : '',
                    'rhost':'',
                    'others':'',
                }
            match_step2 = re.search(pattern_step2, line)
            if match_step2:
                #print("common_messages:", log_dict['common_messages'])
                if log_dict.get('common_messages') == 'pam_unix(sshd:session)':
                    log_dict['type_session'] = match_step2.group(2)
                    log_dict['user'] = match_step2.group(3)
                    log_dict['uid_target'] = match_step2.group(4)
                    log_dict['uid_init'] = match_step2.group(5)
                elif log_dict.get('common_messages') == 'Invalid user':
                    log_dict['user'] = match_step2.group(2)
                    log_dict['IP_user'] = match_step2.group(3)
                    log_dict['port_user'] = match_step2.group(4)
                else :
                    log_dict['others'] = match_step2.group(1)
            print(log_dict)        
        # Entraînez le modèle
        #pipeline_model.fit(X, y)
        
        """#logs = file.read()
        # Traitez les logs selon vos besoins
        # ...
        matches = []
        # Lire chaque ligne du fichier journal
        for line in file.readlines():
            # Rechercher les correspondances dans chaque ligne
            #print(line)
            match = re.match(pattern, line)
            if match:
                # Ajouter les correspondances à la liste
                matches.append(match.groupdict())
            #print(matches)
        df = pd.DataFrame(matches)
        # Afficher le dataframe
        print(df)"""

Jun 25 00:06:00 srvpx01 sshd[1721767]: Invalid user tabadmin from 193.123.114.34 port 44220

{'Timestamp': 'Jun 25 00:06:00', 'Node': 'srvpx01', 'Event': 'sshd', 'sshd_id': '1721767', 'common_messages': 'Invalid user', 'type_session': '', 'type_accepted': '', 'type_received': '', 'type_connection': '', 'type_fail': '', 'type_auth': '', 'user': None, 'IP_user': None, 'port_user': None, 'logname': '', 'uid_target': '', 'uid_init': '', 'euid': '', 'tty': '', 'ruser': '', 'rhost': '', 'others': ''}
Jun 25 00:06:00 srvpx01 sshd[1721767]: pam_unix(sshd:auth): check pass; user unknown

{'Timestamp': 'Jun 25 00:06:00', 'Node': 'srvpx01', 'Event': 'sshd', 'sshd_id': '1721767', 'common_messages': 'pam_unix(sshd:auth)', 'type_session': '', 'type_accepted': '', 'type_received': '', 'type_connection': '', 'type_fail': '', 'type_auth': '', 'user': '', 'IP_user': '', 'port_user': '', 'logname': '', 'uid_target': '', 'uid_init': '', 'euid': '', 'tty': '', 'ruser': '', 'rhost': '', 'others': ''}
Jun 25

KeyboardInterrupt: 

pam_unix session

In [ ]:
pattern = r'(\w+\s\d+\s\d{2}:\d{2}:\d{2})\s(\w+)\s(\w+)(\[(\d+)\]|):\s(\w+)\((\w+)\:(\w+)\):\s\w+\s(\w+)\s\w+\s\w+\s(\w+)\((\w+=\d+)\)\s\w+\s\((\w+=\d+)\)'

#\sfor\suser\s(\w+)\((\w+=\d+)\)\sby\s\((\w+=\d+)\)
# Exemple d'utilisation
log_line = 'Jun 23 15:38:15 srvpx02 sshd[622247]: pam_unix(sshd:session): session opened for user root(uid=0) by (uid=0)'

matchu = re.search(pattern, log_line)
if matchu:
    timestamp = matchu.group(1)
    hostname = matchu.group(2)
    process_name = matchu.group(3)
    process_id = matchu.group(5)
    session_info = matchu.group(6)
    action_1 = matchu.group(7)
    action_2 = matchu.group(8)
    action_3 = matchu.group(9)
    username = matchu.group(10)
    user_info = matchu.group(11)
    initiator_info = matchu.group(12)
    print("Timestamp:", timestamp)
    print("Hostname:", hostname)
    print("Process Name:", process_name)
    print("Process ID:", process_id)
    print("Session Info:", session_info)
    print("Action 1:", action_1)
    print("Action 2:", action_2)
    print("Action 3:", action_3)
    print("Username:", username)
    print("User info:", user_info)
    print("Initiator Info:", initiator_info)

Timestamp: Jun 23 15:38:15
Hostname: srvpx02
Process Name: sshd
Process ID: 622247
Session Info: pam_unix
Action 1: sshd
Action 2: session
Action 3: opened
Username: root
User info: uid=0
Initiator Info: uid=0


In [ ]:
common_messages = [
    r'Accepted',
    r'pam_unix\(.+\): session (?:opened|closed) for user .+',
    r'Failed',
    r'Invalid user .+',
    r'Authentication failure',
    r'refused connect from .+',
    r'error: maximum authentication attempts exceeded',
    r'Connection closed by .+',
    r'useradd',
    r'userdel',
    r'passwd',
]



# Concaténation des messages en une seule expression régulière
pattern = r'({})'.format('|'.join(common_messages))

# Exemple d'utilisation avec un fichier auth.log
with open('auth.log', 'r') as file:
    for line in file.readlines():
        match = re.search(pattern, line)
        if match:
            print("Matched message:", match.group(0))

FileNotFoundError: [Errno 2] No such file or directory: 'auth.log'